Mechanism Hypotheses:
1. Lateral exchange w/ surrounding water. For a cyclone moving south, water with low CHL enters and dilutes/reduces CHL concentration. 
2. Eddy-wind interaction. Ekman pumping drives upwelling in anticyclones and downwelling in cyclones. But if this is the case, then we should expect a change/response at the centers first.
3. Seasonal changes and mixed layer depth changes. 

In [ ]:
from typing import cast

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr
from IPython.display import display
from matplotlib.axes import Axes
from matplotlib.figure import Figure
from matplotlib.ticker import MaxNLocator

from eddy_tracking.config import PROJECT_ROOT, load_config
from eddy_tracking.preprocess.streamline import compute_signed_distance_grid_km, compute_signed_distance_km, index_centerlines_by_date
from eddy_tracking.preprocess.tracks import load_date_eddy_index, load_track_observations
from eddy_tracking.utils.analysis_period import restrict_to_analysis_period
from eddy_tracking.utils.composite_regions import average_regions, list_composites, locate_interior

EXPERIMENT = 'gulf_stream_20240305_20260531'
DATA_DIR = PROJECT_ROOT / 'data' / EXPERIMENT
cfg = load_config(EXPERIMENT)
NEAR_AXIS_KM = 150
identity_columns = ['polarity', 'track_id']
polarity_names = ('cyclone', 'anticyclone')
target_classes = {'cyclone': 'NS', 'anticyclone': 'SN'}
polarity_colors = {'cyclone': '#2166ac', 'anticyclone': '#b2182b'}
polarity_labels = {'cyclone': 'Target cyclones', 'anticyclone': 'Target anticyclones'}
polarity_sides = {'cyclone': ('north side', 'south side'), 'anticyclone': ('south side', 'north side')}
panel_letters = 'abcdefghijklmnop'
plankton_settings = cfg['collocate_plankton']

eddy_tracks = restrict_to_analysis_period(pd.read_parquet(DATA_DIR / 'silver/gulf_stream/eddy_movement.parquet'), cfg)
# Only the eddies first detected on one side of the axis and last detected on the other: cyclones from north to south, anticyclones from south to north
target_tracks = eddy_tracks.loc[eddy_tracks['movement'].eq(cast(pd.Series, eddy_tracks['polarity']).map(target_classes)), identity_columns + ['birth_date', 'death_date']]
target_keys = set(zip(target_tracks['polarity'], target_tracks['track_id']))
date_index = load_date_eddy_index(EXPERIMENT)
observations = load_track_observations(EXPERIMENT).drop(columns=['contour_lon', 'contour_lat']).merge(target_tracks[identity_columns], on=identity_columns).sort_values(identity_columns + ['date'], ignore_index=True)
centerlines = index_centerlines_by_date(pd.read_parquet(DATA_DIR / 'silver/gulf_stream/streamline.parquet'))
# Distance from the daily axis toward the side a target eddy ends on, so positive is south of the axis for a cyclone and north of it for an anticyclone
observations['across_km'] = cast(pd.Series, observations['polarity']).map({'cyclone': -1, 'anticyclone': 1}) * [
    compute_signed_distance_km(centerlines[day.date()].lon, centerlines[day.date()].lat, lon, lat)[0]
    for day, lon, lat in zip(observations['date'], observations['center_lon'], observations['center_lat'])
]


def add_phase_position(table: pd.DataFrame) -> pd.DataFrame:
    # The departure is the first day an eddy is farther than NEAR_AXIS_KM past the axis; a target eddy that never gets that far has no second phase and drops out here
    departure = cast(pd.Series, observations.loc[observations['across_km'].gt(NEAR_AXIS_KM)].groupby(identity_columns)['date'].min()).rename('departure_date').reset_index()
    table = table.merge(target_tracks, on=identity_columns, suffixes=('', '_track')).merge(departure, on=identity_columns)
    before = table['date'] < table['departure_date']
    start = table['birth_date'].where(before, table['departure_date'])
    end = table['departure_date'].where(before, table['death_date'])
    # 0 to 1 is the time on the original side, from the first detection to the departure, and 1 to 2 the time on the new side, up to the last detection; a composite midpoint can fall outside the track, hence the clip
    table['phase_position'] = ((table['date'] - start).dt.days / (end - start).dt.days.clip(lower=1)).clip(0, 1) + ~before
    return table


plt.rcParams.update({
    'font.family': 'sans-serif', 'font.sans-serif': ['Arial', 'Helvetica', 'DejaVu Sans'],
    'mathtext.fontset': 'custom', 'mathtext.rm': 'Arial', 'mathtext.it': 'Arial:italic', 'mathtext.bf': 'Arial:bold',
    # matplotlib 3.11 warns about its default calligraphic face, Apple Chancery, even though no label uses it
    'mathtext.cal': 'Arial',
    'font.size': 8, 'axes.titlesize': 8, 'axes.labelsize': 8, 'xtick.labelsize': 8, 'ytick.labelsize': 8, 'legend.fontsize': 8,
    'axes.linewidth': 0.6, 'xtick.major.width': 0.6, 'ytick.major.width': 0.6, 'xtick.major.size': 2.5, 'ytick.major.size': 2.5,
    'axes.spines.top': False, 'axes.spines.right': False, 'legend.frameon': False,
    'figure.dpi': 150, 'savefig.dpi': 300,
})

## PACE Rrs spectral shape

In [ ]:
composites = list_composites(DATA_DIR / 'bronze/sst', date_index, target_keys)
rrs_files = {path.name.split('.')[1]: path for path in (DATA_DIR / 'bronze/pace_l3_8d').glob('*.nc')}
with xr.open_dataset(next(iter(rrs_files.values()))) as ds:
    wavelengths = ds['wavelength'].to_numpy()
    rrs_lon = ds['lon'].to_numpy()
    rrs_lat = ds['lat'].to_numpy()
rrs_lon_grid, rrs_lat_grid = np.meshgrid(rrs_lon, rrs_lat)
visible = np.flatnonzero((wavelengths >= 400) & (wavelengths <= 700))
rrs_rows = []
for path, midpoint, eddies, _ in composites:
    with xr.open_dataset(rrs_files[path.name.split('.')[1]]) as ds:
        field = ds['Rrs'].isel(wavelength=visible).to_numpy()  # (n_lat, n_lon, n_visible)
    centerline = centerlines[midpoint.date()]
    axis_km = compute_signed_distance_grid_km(centerline.lon, centerline.lat, rrs_lon, rrs_lat)
    for eddy in eddies:
        half_chord = (
            np.sin(np.radians(rrs_lat_grid - eddy.center_lat) / 2) ** 2
            + np.cos(np.radians(rrs_lat_grid)) * np.cos(np.radians(eddy.center_lat)) * np.sin(np.radians(rrs_lon_grid - eddy.center_lon) / 2) ** 2
        )
        radius_units = 2 * 6371 * np.arcsin(np.sqrt(half_chord)) / eddy.radius_km
        # Each side's water is the band 150 to 450 km from the axis within 4 degrees of longitude of the eddy, with the eddy and its 3 R surroundings cut out
        band = (np.abs(rrs_lon_grid - eddy.center_lon) <= 4) & (radius_units > 3)
        regions = {
            'interior': locate_interior(rrs_lon, rrs_lat, eddy.contour_lon, eddy.contour_lat),
            'north': band & (axis_km >= 150) & (axis_km <= 450), 'south': band & (axis_km <= -150) & (axis_km >= -450),
        }
        means = average_regions(field, regions, plankton_settings['min_pixels'], plankton_settings['min_coverage'])
        # Each spectrum divided by its own mean, so the comparison follows the shape and not the brightness
        shapes = {region: mean / mean.mean() for region, mean in means.items()}
        original, new = (shapes['north'], shapes['south']) if eddy.polarity == 'cyclone' else (shapes['south'], shapes['north'])
        across = new - original
        # The interior shape projected on the straight line from the original-side shape (0) to the new-side shape (1); mixing of the two waters keeps it on that line, so the distance off the line, in units of the line's length, shows change that mixing cannot explain
        position = np.dot(shapes['interior'] - original, across) / np.dot(across, across)
        off_line = np.linalg.norm(shapes['interior'] - original - position * across) / np.linalg.norm(across)
        rrs_rows.append({'polarity': eddy.polarity, 'track_id': eddy.track_id, 'date': midpoint, 'position': position, 'off_line': off_line, 'interior': shapes['interior'], 'original': original, 'new': new})
rrs = add_phase_position(restrict_to_analysis_period(pd.DataFrame(rrs_rows), cfg))
complete = rrs.dropna(subset=['position'])

AGE_BIN_WIDTH = 0.5
MIN_BIN_COMPOSITES = 3
n_age_bins = round(2 / AGE_BIN_WIDTH)
side_water_colors = {'original': '#1b9e77', 'new': '#222222'}
# Mean shapes over the eddy composites of each polarity: the water on each side, and the interior binned by its position in the two phases (halves of each side), fading in from transparent (early) to opaque (late) within each side, so the dash style carries the side and the opacity the age within it; dashed while still on the original side, solid once on the new side
rrs_fig, rrs_axes = cast(tuple[Figure, np.ndarray], plt.subplots(2, 1, figsize=(6.69, 6.4), layout='constrained'))
for ax, letter, polarity in zip(cast(list[Axes], rrs_axes.tolist()), panel_letters, polarity_names):
    members = complete.loc[complete['polarity'].eq(polarity)]
    original_side, new_side = (side.split()[0] for side in polarity_sides[polarity])
    color = polarity_colors[polarity]
    for column, side in (('original', original_side), ('new', new_side)):
        ax.plot(wavelengths[visible], np.stack(members[column].to_list()).mean(axis=0), color=side_water_colors[column], linewidth=2, label=f'{side.capitalize()}-side water ({column})', zorder=5)
    # phase_position runs 0 to 1 on the original side and 1 to 2 on the new side; a composite exactly at 2 joins the last bin
    age_bin = np.minimum((members['phase_position'] / AGE_BIN_WIDTH).astype(int), n_age_bins - 1)
    for index in range(n_age_bins):
        rows = members.loc[age_bin == index]
        on_new_side = index >= n_age_bins // 2
        low = round(index * AGE_BIN_WIDTH - on_new_side, 1)
        label = f'Interior, {"new" if on_new_side else "original"} side {low:g} to {low + AGE_BIN_WIDTH:g} ({len(rows)})'
        if len(rows) < MIN_BIN_COMPOSITES:
            # Keep the legend entry so a missing bin is visible rather than silently skipped
            ax.plot([], [], color=color, alpha=0, label=f'{label}, too few')
            continue
        ax.plot(
            wavelengths[visible], np.stack(rows['interior'].to_list()).mean(axis=0), color=color, alpha=0.3 + 0.7 * (index % (n_age_bins // 2)) / max(n_age_bins // 2 - 1, 1),
            linestyle='-' if on_new_side else (0, (3, 1.5)), linewidth=1.5, label=label, zorder=4,
        )
    ax.set_title(rf'$\bf{{({letter})}}$ {polarity_labels[polarity]}, {original_side} side to {new_side} side (n = {members["track_id"].nunique()})', loc='left', color=color)
    ax.set_xlim(400, 700)
    ticks = cast(np.ndarray, MaxNLocator(nbins=5, steps=[1, 2, 2.5, 5, 10]).tick_values(0, ax.dataLim.y1))
    ax.yaxis.set_ticks(ticks.tolist())
    ax.set_ylim(0, ticks[-1])
    ax.grid(axis='y', color='#e5e5e5', linewidth=0.5)
    ax.set_axisbelow(True)
    ax.set_xlabel('Wavelength (nm)')
    ax.set_ylabel('Rrs / mean Rrs')
    ax.legend(loc='center left', bbox_to_anchor=(1.01, 0.5), handlelength=2.2, title='Mean shape (composites)', title_fontsize=8, alignment='left', fontsize=7)
plt.show()
display(complete.assign(age_bin=np.minimum((complete['phase_position'] / AGE_BIN_WIDTH).astype(int), n_age_bins - 1)).groupby(['polarity', 'age_bin']).agg(composites=('date', 'size'), eddies=('track_id', 'nunique')).unstack('age_bin'))
